In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path(__file__).parent if '__file__' in dir() else Path().resolve()

df = pd.read_csv(
    BASE / "../../../data/processed/2024/datos_limpios.csv",
    low_memory=False
)

# 1. Producción total por pozo en el año
# Por qué: identificar los pozos más productivos
print("=== TOP 10 POZOS POR PRODUCCIÓN DE PETRÓLEO ===")
por_pozo = df.groupby(['idpozo', 'empresa', 'provincia', 'tipo_de_recurso'])[
    ['prod_pet', 'prod_gas', 'prod_agua', 'iny_agua', 'iny_gas', 'tef']
].sum()
por_pozo = por_pozo.sort_values('prod_pet', ascending=False)
print(por_pozo.head(10))

# 2. Eficiencia: relación producción vs inyección
# Por qué: un pozo que necesita mucha inyección para producir poco es ineficiente
print("\n=== EFICIENCIA DE POZOS ===")
por_pozo['inyeccion_total'] = por_pozo['iny_agua'] + por_pozo['iny_gas']
por_pozo['eficiencia_pet'] = por_pozo.apply(
    lambda x: x['prod_pet'] / x['inyeccion_total'] 
    if x['inyeccion_total'] > 0 else None, axis=1
)

# Solo pozos que producen petróleo y usan inyección
pozos_con_inyeccion = por_pozo[
    (por_pozo['prod_pet'] > 0) & 
    (por_pozo['inyeccion_total'] > 0)
].copy()

print(f"Pozos que producen y usan inyección: {len(pozos_con_inyeccion)}")
print("\nTop 10 más eficientes:")
print(pozos_con_inyeccion.sort_values('eficiencia_pet', ascending=False).head(10))
print("\nTop 10 menos eficientes (más costosos):")
print(pozos_con_inyeccion.sort_values('eficiencia_pet', ascending=True).head(10))

# 3. Declive de producción a lo largo del año
# Por qué: detectar pozos que están perdiendo productividad
print("\n=== DECLIVE DE PRODUCCIÓN ===")
declive = df.groupby(['idpozo', 'mes'])['prod_pet'].sum().unstack(fill_value=0)

# Comparamos primer semestre vs segundo semestre
declive['primer_semestre'] = declive[[1,2,3,4,5,6]].sum(axis=1)
declive['segundo_semestre'] = declive[[7,8,9,10,11,12]].sum(axis=1)
declive['variacion'] = (
    (declive['segundo_semestre'] - declive['primer_semestre']) / 
    declive['primer_semestre'].replace(0, float('nan')) * 100
).round(2)

pozos_en_declive = declive[declive['variacion'] < -30].sort_values('variacion')
pozos_creciendo = declive[declive['variacion'] > 30].sort_values('variacion', ascending=False)

print(f"Pozos con caída mayor al 30%: {len(pozos_en_declive)}")
print(f"Pozos con crecimiento mayor al 30%: {len(pozos_creciendo)}")
print("\nTop 10 pozos en mayor declive:")
print(pozos_en_declive.head(10)[['primer_semestre', 'segundo_semestre', 'variacion']])
print("\nTop 10 pozos con mayor crecimiento:")
print(pozos_creciendo.head(10)[['primer_semestre', 'segundo_semestre', 'variacion']])

# 4. Guardar resultados para Power BI
por_pozo.reset_index().to_csv(
    BASE / "../../../data/processed/2024/analisis_pozos.csv",
    index=False
)
declive.reset_index().to_csv(
    BASE / "../../../data/processed/2024/declive_pozos.csv",
    index=False
)

print("\nArchivos guardados correctamente")

=== TOP 10 POZOS POR PRODUCCIÓN DE PETRÓLEO ===
                                                                  prod_pet  \
idpozo empresa                    provincia tipo_de_recurso                  
164941 VISTA ENERGY ARGENTINA SAU Neuquén   NO CONVENCIONAL  155502.480000   
164942 VISTA ENERGY ARGENTINA SAU Neuquén   NO CONVENCIONAL   86605.735000   
164204 TOTAL AUSTRAL S.A.         Neuquén   NO CONVENCIONAL   71355.786279   
165093 KILWER S.A.                Neuquén   NO CONVENCIONAL   70574.010200   
164205 TOTAL AUSTRAL S.A.         Neuquén   NO CONVENCIONAL   68127.048569   
165097 SHELL ARGENTINA S.A.       Neuquén   NO CONVENCIONAL   67050.970000   
164127 VISTA ENERGY ARGENTINA SAU Neuquén   NO CONVENCIONAL   67032.091000   
164799 YPF S.A.                   Neuquén   NO CONVENCIONAL   62968.410000   
164713 VISTA ENERGY ARGENTINA SAU Neuquén   NO CONVENCIONAL   62419.261000   
164802 YPF S.A.                   Neuquén   NO CONVENCIONAL   61163.450000   

               